In [ ]:
#!/usr/bin/env python
"""
Pipeline for Pocket Identification, Tunnel Finding, and Docking

This script:
  • Loads a protein structure (and optionally an MD trajectory) using MDAnalysis,
    retaining only protein atoms.
  • Identifies a pocket center by one of several methods:
       - "atom": Use the active-site atom’s position.
       - "ligand_frame0": Use the ligand’s center-of–mass in frame 0.
       - "clustering": Use DBSCAN clustering on protein atoms near the active site to 
                     determine the pocket (returns both the centroid and the pocket atoms).
  • Uses the identified pocket center for visualization.
  • Builds a 3D grid over the entire protein and computes clearance.
  • Finds a tunnel from the active site (or nearest accessible grid point) to the grid boundary
    using a maximum–bottleneck algorithm.
  • Analyzes tunnel geometry.
  • Computes both a lower-bound energy profile (via full docking) and an upper-bound energy profile 
    (via optimization) along the tunnel.
  • Visualizes the computed tunnel, pocket, and ligand trajectories on the protein.
  • Ingestion functions use OpenBabel (pybel) to convert ligand files (SDF/MOL2) and protein files (PDB)
    to PDBQT (using the "-r" option for the protein to obtain a rigid molecule).
  
Requirements:
  - MDAnalysis, SciPy, NumPy, matplotlib, py3Dmol, scikit-learn, autodock-vina Python API,
    and OpenBabel (with pybel and openbabel).
"""

import os, tempfile
import numpy as np
import MDAnalysis as mda
from MDAnalysis.lib import distances
import matplotlib.pyplot as plt
import heapq
from sklearn.cluster import DBSCAN
from vina import Vina
import py3Dmol
from openbabel import pybel
from openbabel import openbabel as ob
from multiprocessing import Pool, cpu_count
import numpy as np
import heapq
from scipy.spatial import Voronoi, KDTree, ConvexHull

### Ingestion Functions ###
def convert_to_pdbqt_ligand(input_file, output_file=None):
    ext = os.path.splitext(input_file)[1].lower()[1:]
    if output_file is None:
        base = os.path.splitext(input_file)[0]
        output_file = base + ".pdbqt"
    mols = list(pybel.readfile(ext, input_file))
    if not mols:
        raise ValueError(f"No molecules found in {input_file}")
    mol = mols[0]
    mol.addh()       # Add hydrogens (for pH ~7.4)
    mol.make3D()     # Generate 3D geometry
    mol.calccharges(model='gasteiger')
    mol.write("pdbqt", output_file, overwrite=True)
    return output_file

def convert_to_pdbqt_protein(input_file, output_file=None):
    if os.path.splitext(input_file)[1].lower() != ".pdb":
        raise ValueError("Protein input must be a PDB file.")
    if output_file is None:
        base = os.path.splitext(input_file)[0]
        output_file = base + "_rigid.pdbqt"
    conv = ob.OBConversion()
    conv.SetInFormat("pdb")
    conv.SetOutFormat("pdbqt")
    conv.AddOption("r", ob.OBConversion.OUTOPTIONS)  # rigid output
    obmol = ob.OBMol()
    conv.ReadFile(obmol, input_file)
    obmol.CorrectForPH()  # optional
    obmol.AddHydrogens()  # ensure hydrogens are present
    conv.WriteFile(obmol, output_file)
    return output_file

def universe_frame_to_pdbqt(u, output_file):
    temp_pdb = tempfile.NamedTemporaryFile(delete=False, suffix=".pdb").name
    with mda.Writer(temp_pdb) as W:
        W.write(u.atoms)
    pdbqt_file = convert_to_pdbqt_protein(temp_pdb, output_file)
    os.remove(temp_pdb)
    return pdbqt_file

def ingest_files(protein_input, ligand_input):
    protein_ext = os.path.splitext(protein_input)[1].lower()
    ligand_ext = os.path.splitext(ligand_input)[1].lower()
    if protein_ext != ".pdbqt":
        protein_pdbqt = convert_to_pdbqt_protein(protein_input)
    else:
        protein_pdbqt = protein_input
    if ligand_ext != ".pdbqt":
        ligand_pdbqt = convert_to_pdbqt_ligand(ligand_input)
    else:
        ligand_pdbqt = ligand_input
    return protein_pdbqt, ligand_pdbqt

### Pocket Identification Functions ###
def load_protein(pdb_file, trajectory_file=None):
    if trajectory_file is not None:
        u = mda.Universe(pdb_file, trajectory_file)
    else:
        u = mda.Universe(pdb_file)
    return u

def get_protein_only(u):
    return u.select_atoms("protein")

def pocket_atom(active_atom):
    return active_atom.position

def pocket_ligand_frame0(u, ligand_sel):
    ligand = u.select_atoms(ligand_sel)
    if len(ligand) == 0:
        raise ValueError("No ligand found using the provided selection in frame 0.")
    return ligand.center_of_mass()

def pocket_clustering(u, active_atom, search_radius=12.0, eps=3.0, min_samples=5):
    protein = get_protein_only(u)
    nearby = protein.select_atoms("around %f index %d" % (search_radius, active_atom.index))
    if active_atom not in nearby:
        active_ag = u.select_atoms("index %d" % active_atom.index)
        nearby = nearby.union(active_ag)
    coords = nearby.positions
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(coords)
    labels = db.labels_
    unique_labels = set(labels)
    if -1 in unique_labels:
        unique_labels.remove(-1)
    if not unique_labels:
        raise ValueError("No clusters found; try adjusting eps/min_samples.")
    centroids = {label: coords[labels == label].mean(axis=0) for label in unique_labels}
    active_pos = active_atom.position
    best_label = min(centroids.keys(), key=lambda l: np.linalg.norm(centroids[l] - active_pos))
    pocket_atoms = nearby[labels == best_label]
    return centroids[best_label], pocket_atoms

def get_pocket_center(u, method, active_site_atom_sel, ligand_sel, **kwargs):
    if method == "atom":
        active_atoms = u.select_atoms(active_site_atom_sel)
        if len(active_atoms) == 0:
            raise ValueError("No active site atom found!")
        return pocket_atom(active_atoms[0]), None
    elif method == "ligand_frame0":
        u.trajectory[0]
        return pocket_ligand_frame0(u, ligand_sel), None
    elif method == "clustering":
        active_atoms = u.select_atoms(active_site_atom_sel)
        if len(active_atoms) == 0:
            raise ValueError("No active site atom found!")
        return pocket_clustering(u, active_atoms[0], **kwargs)
    else:
        raise ValueError("Unknown pocket method: choose from 'atom', 'ligand_frame0', or 'clustering'.")

### Grid and Tunnel Functions ###
def compute_voronoi(u):
    protein = get_protein_only(u)
    coords = protein.positions
    vor = Voronoi(coords)
    return vor

def compute_clearance_voronoi(vor, protein_coords, probe_radius):
    tree = KDTree(protein_coords)
    clearances = {}
    for i, v in enumerate(vor.vertices):
        d, _ = tree.query(v, k=1)
        clearances[tuple(v)] = d - probe_radius
    return clearances

def build_voronoi_graph(vor, clearances):
    graph = {}
    for ridge in vor.ridge_vertices:
        if -1 in ridge:
            continue  # Ignore infinite ridges
        v1, v2 = vor.vertices[ridge[0]], vor.vertices[ridge[1]]
        if tuple(v1) in clearances and tuple(v2) in clearances:
            clearance = min(clearances[tuple(v1)], clearances[tuple(v2)])
            graph.setdefault(tuple(v1), []).append((tuple(v2), clearance))
            graph.setdefault(tuple(v2), []).append((tuple(v1), clearance))
    return graph

def find_closest_voronoi_vertex(active_atom, vor, clearances):
    active_pos = np.array(active_atom.position)
    tree = KDTree(list(clearances.keys()))
    _, idx = tree.query(active_pos, k=1)
    return list(clearances.keys())[idx]

def visualize_voronoi_graph(start, graph):
    view = py3Dmol.view()
    
    # Highlight the start point
    x, y, z = start
    view.addSphere({'center': {'x': x, 'y': y, 'z': z}, 'radius': 0.5, 'color': 'red'})
    
    # Draw all connected points
    for point, neighbors in graph.items():
        px, py, pz = point
        view.addSphere({'center': {'x': px, 'y': py, 'z': pz}, 'radius': 0.3, 'color': 'blue'})
        for neighbor, _ in neighbors:
            nx, ny, nz = neighbor
            view.addCylinder({'start': {'x': px, 'y': py, 'z': pz}, 'end': {'x': nx, 'y': ny, 'z': nz}, 'radius': 0.1, 'color': 'gray'})
    
    view.zoomTo()
    view.show()
    return 

def find_all_tunnels_dijkstra(active_atom, graph, clearances, vor):
    start = find_closest_voronoi_vertex(active_atom, vor, clearances)
    visualize_voronoi_graph(start, graph)
    
    pq = [(-clearances[start], start)]  # Max heap based on clearance
    parent = {start: None}
    best_clearance = {start: clearances[start]}
    tunnels = []
    
    while pq:
        curr_clearance, curr = heapq.heappop(pq)
        curr_clearance = -curr_clearance
        
        if is_boundary_voronoi(curr, graph):
            path = []
            node = curr
            while node:
                path.append(node)
                node = parent.get(node)
            path.reverse()
            tunnels.append(np.array(path))
            continue
        
        for neighbor, clearance in graph.get(curr, []):
            new_clearance = min(curr_clearance, clearance)
            if neighbor not in best_clearance or new_clearance > best_clearance[neighbor]:
                best_clearance[neighbor] = new_clearance
                parent[neighbor] = curr
                heapq.heappush(pq, (-new_clearance, neighbor))
    
    if not tunnels:
        raise ValueError("No tunnels found! Ensure the active site is connected to the convex hull.")
    
    return tunnels

def is_boundary_voronoi(point, graph):
    hull = ConvexHull(np.array(list(graph.keys())))
    return any(np.linalg.norm(point - hull.points[v]) < 1.0 for v in hull.vertices)  # Allow small threshold

def analyze_tunnel_geometry_voronoi(protein_coords, tunnel_paths, probe_radius):
    tree = KDTree(protein_coords)
    all_radii = []
    bottlenecks = []
    
    for path in tunnel_paths:
        radii = [max(tree.query(point, k=1)[0] - probe_radius, 0.0) for point in path]
        all_radii.append(np.array(radii))
        bottlenecks.append(np.min(radii))
    
    return all_radii, bottlenecks

def cluster_tunnels(tunnel_paths, threshold=2.0):
    clusters = []
    for path in tunnel_paths:
        added = False
        for cluster in clusters:
            if np.linalg.norm(path[0] - cluster[0][0]) < threshold:
                cluster.append(path)
                added = True
                break
        if not added:
            clusters.append([path])
    return clusters

### Visualization Functions ###
def visualize_tunnel_py3dmol(tunnel_path, protein_vis_file, pocket_atoms=None, tunnel_radii=None):
    with open(protein_vis_file, 'r') as f:
        pdb_data = f.read()
    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_data, 'pdb')
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    for i, point in enumerate(tunnel_path):
        rad = float(tunnel_radii[i]) if tunnel_radii is not None else 0.5
        view.addSphere({
            'center': {'x': float(point[0]), 'y': float(point[1]), 'z': float(point[2])},
            'radius': rad, 'color': 'red', 'opacity': 0.8
        })
    if pocket_atoms is not None and len(pocket_atoms) > 0:
        for atom in pocket_atoms:
            view.addSphere({
                'center': {'x': float(atom.position[0]),
                           'y': float(atom.position[1]),
                           'z': float(atom.position[2])},
                'radius': 0.5, 'color': 'blue', 'opacity': 0.3
            })
    view.zoomTo()
    view.show()

def visualize_ligand_trajectory(trajectory_results, protein_vis_file, traj_label, color):
    """
    Visualize the ligand trajectory on the protein.
    trajectory_results should be an array of (distance, energy, ligand_pose_str).
    Each valid ligand_pose_str is added as a separate model.
    """
    with open(protein_vis_file, 'r') as f:
        pdb_data = f.read()
    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_data, 'pdb')
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    model_index = 1
    for i, (dist, energy, pose_str) in enumerate(trajectory_results):
        if pose_str is not None and len(pose_str) > 0:
            view.addModel(pose_str, 'pdb')
            view.setStyle({'model': model_index}, {'stick': {'colorscheme': color}})
            view.addLabel(f"{traj_label} {i+1}", {'model': model_index, 'backgroundColor': color, 'fontColor': 'white', 'fontSize': 10})
            model_index += 1
    view.zoomTo()
    view.show()

### Main Workflow ###
def main():
    protein_input = "1be0.pdb"          # Protein input file (for visualization)
    ligand_input = "EtOH.sdf"           # Ligand input file (SDF/MOL2)
    traj_file = None                   # Static structure
    
    active_site_atom_sel = "resid 124 and name OD2"
    pocket_method = "clustering"
    grid_spacing = 1.0
    probe_radius = 1.4
    grid_margin = 5.0
    
    # Ingest docking files (PDBQT conversion)
    protein_dock, ligand_dock = ingest_files(protein_input, ligand_input)
    
    # Load protein for analysis/visualization
    u = load_protein(protein_input, traj_file)
    active_atoms = u.select_atoms(active_site_atom_sel)
    if len(active_atoms) == 0:
        raise ValueError("No active site atom found!")
    active_atom = active_atoms[0]
    
    if pocket_method == "clustering":
        pocket_center, pocket_atoms = get_pocket_center(u, method=pocket_method,
                                                        active_site_atom_sel=active_site_atom_sel,
                                                        ligand_sel="resname LIG",
                                                        search_radius=12.0, eps=3.0, min_samples=5)
    else:
        pocket_center, pocket_atoms = get_pocket_center(u, method=pocket_method,
                                                        active_site_atom_sel=active_site_atom_sel,
                                                        ligand_sel="resname LIG")
    pocket_center = np.array(pocket_center, dtype=float)
    
    vor = compute_voronoi(u)
    protein_coords = get_protein_only(u).positions
    clearances = compute_clearance_voronoi(vor, protein_coords, probe_radius)
    graph = build_voronoi_graph(vor, clearances)
    
    try:
        tunnel_paths = tunnel_paths = find_all_tunnels_dijkstra(active_atom, graph, clearances, vor)
    except ValueError as e:
        print("Tunnel finding error:", e)
        return
    
    radii, bottlenecks = analyze_tunnel_geometry_voronoi(protein_coords, tunnel_paths, probe_radius)
    clusters = cluster_tunnels(tunnel_paths)
    
    dists_profile = np.linspace(0, np.linalg.norm(tunnel_path[-1] - tunnel_path[0]), len(radii))
    plt.figure(figsize=(8, 4))
    plt.plot(dists_profile, 2 * radii, marker='o')
    plt.xlabel("Distance along tunnel (Å)")
    plt.ylabel("Estimated tunnel diameter (Å)")
    plt.title("Tunnel Geometry Profile")
    plt.grid(True)
    plt.show()
    

if __name__ == "__main__":
    main()


*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is 1be0.pdb)

